# Simulation of LUCJ circuits

In [1]:
import time

import matplotlib.pyplot as plt; plt.rcParams.update({"font.family": "serif", "font.size": 12})
import numpy as np

import ffsim

import openfermion as of

from pyscf import ao2mo, tools, cc

import qiskit
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import CouplingMap
from qiskit.quantum_info import Statevector, SparsePauliOp

## Circuit construction

Following https://github.com/jrm874/sqd_data_repository/blob/main/experiments/circuit_generation_template/circuit_generation_template.py.

In [ ]:
fcidump_filename = "integrals/4Fe-4S/fcidump_Fe4S4_MO.txt"  # Note: Requires data from above repo.


mf_as = tools.fcidump.to_scf(fcidump_filename)
h1e = mf_as.get_hcore()

In [ ]:
num_orb = h1e.shape[0]  # TODO: Verify.
num_elec_a = num_orb // 2
num_elec_b = num_orb // 2

h2e = ao2mo.restore(1, mf_as._eri, num_orb)

In [ ]:
ccsd = cc.CCSD(mf_as).run()
t1 = ccsd.t1
t2 = ccsd.t2


n_reps = 1
alpha_alpha_indices = [(p, p + 1) for p in range(num_orb - 1)]
alpha_beta_indices = [(p, p) for p in range(0, num_orb, 4)]

ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices),
)

nelec = (num_elec_a, num_elec_b)

# create an empty quantum circuit
qubits = qiskit.QuantumRegister(2 * num_orb, name="q")
circuit = qiskit.QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(num_orb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)

In [ ]:
coupling_map = CouplingMap.from_grid(
    num_rows=int(np.ceil(np.sqrt(2 * num_orb))),
    num_columns=int(np.ceil(np.sqrt(2 * num_orb)))
)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

In [ ]:
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
        backend=backend,
        norb=num_orb,
        connectivity="square",
        interaction_pairs=(alpha_alpha_indices, alpha_beta_indices),
        optimization_level=3,
    )

In [ ]:
compiled = pass_manager.run(circuit)

In [ ]:
print(f"Number of qubits: {compiled.num_qubits}")
print(f"Gate counts: {compiled.count_ops()}")

In [ ]:
compiled.draw(fold=-1)

## Observable definition

### From the Hamiltonian

In [ ]:
try:
    constant = mf_as.energy_nuc()
except Exception:
    constant = 0.0

h2e_of = np.einsum("prqs->pqrs", h2e)

molecular_hamiltonian = of.InteractionOperator(
    constant=constant,
    one_body_tensor=h1e,
    two_body_tensor=0.5 * h2e_of,
)

qubit_op = of.jordan_wigner(of.get_fermion_operator(molecular_hamiltonian))
print(f"Hamiltonian acts on {of.utils.count_qubits(qubit_op)} qubit(s) and has {len(qubit_op.terms)} Pauli term(s).")

In [ ]:
n_circuit_qubits = compiled.num_qubits

pauli_list = []
for i, (term, coeff) in enumerate(qubit_op.terms.items()):
    print(f"On term {i}", end="\r")
    pauli_str = ['I'] * n_circuit_qubits
    for qubit_idx, pauli_char in term:
        pauli_str[qubit_idx] = pauli_char
    pauli_str = ''.join(reversed(pauli_str))
    pauli_list.append((pauli_str, coeff))
hamiltonian = SparsePauliOp.from_list(pauli_list)

In [ ]:
# TODO: Fix the mismatch between Hamiltonian qubit number and circuit qubit number.
hamiltonian = hamiltonian[1:]  # Remove identity.
hamiltonian = hamiltonian.chop(1e-6)

sorted_indices = np.argsort(-np.abs(hamiltonian.coeffs))
hamiltonian = hamiltonian[sorted_indices]

plt.semilogy(
    np.abs(hamiltonian.coeffs)
)

In [ ]:
observables = [SparsePauliOp(op) for op in hamiltonian.paulis[100:110]]

### Weight two operators

In [ ]:
observables = [
    SparsePauliOp("ZZ" + "I" * (compiled.num_qubits - 2)),
    SparsePauliOp("I" * (compiled.num_qubits // 2) + "ZZ" + "I" * (compiled.num_qubits // 2 - 2)),
    SparsePauliOp("I" * (compiled.num_qubits - 2) + "ZZ")
]
print(observables)

## Varying weight operators

In [ ]:
observables = [
    SparsePauliOp("I" * (i + 1) + "Z" * (compiled.num_qubits - i - 1)) for i in range(compiled.num_qubits - 2, -2, -1)
]
observables

## Exact values

In [ ]:
if compiled.num_qubits <= 20:
    expectation_values_exact = []
    for observable in observables:
        statevector = Statevector(compiled)
        sv_expectation_value = statevector.expectation_value(observable).real
        print(sv_expectation_value)
        expectation_values_exact.append(sv_expectation_value)

## Heisenberg simulation

In [ ]:
from propaq.datatypes.majorana import MajoranaMonomial

from propaq.propagators import MajoranaPropagator
from propaq.circuits import MajoranaCircuit 
from propaq.noise import UniformNoiseModel, truncation
from propaq.noise import TruncationPolicy 

from propaq.datatypes import MajoranaTermSum

In [ ]:
mc = MajoranaCircuit.from_qiskit(compiled.copy(), n_modes=2 * compiled.num_qubits)

In [ ]:
damping: float = 0.001
cutoff: float = 1e-6
prop = MajoranaPropagator(
    UniformNoiseModel(damping=damping),
    TruncationPolicy(weight_cutoff=100000, coeff_cutoff=cutoff),
    n_threads=20,
    progress_bar=True,
)

In [ ]:
results = []
times = []
for observable in observables[:]:
    print("On observable", observable)
    observable_mts = MajoranaTermSum.from_sparse_pauli_op(observable)
    print(f"Observable has {len(observable_mts.items())} Majorana monomial(s)")
    for m in observable_mts.items():
        print(m)
    start = time.monotonic()
    results.append(prop.expectation_value(observable_mts, mc, fock_state=0))
    times.append(time.monotonic() - start)
    print("Elapsed time:", times[-1], "seconds")
    print(results[-1].expectation_value)

In [ ]:
# colors = ["std:blue", "std:green", "std:orange"]
for i, result in enumerate(results):
    # TODO: Match the label to the actual observable (Pauli type and weight) and format better.
    plt.semilogy(result.n_terms, "--s", alpha=0.55, ms=5, label="Z " + ",".join(map(str, observables[i].to_sparse_list()[0][1])))

plt.xlabel("Gate index")
plt.ylabel("Number of Majorana monomials")
plt.title(f"{fcidump_filename}\n {len(compiled.qubits)} qubits, {n_reps} LUCJ layer(s), noise damping = {damping}, cutoff = {cutoff}\n{compiled.count_ops()}", size=10)
plt.legend();
plt.tight_layout()
# plt.savefig("science2025lucj_2layers_FeS_weight_1_observables.pdf");